In [ ]:
import json
import matplotlib.pyplot as plt

In [ ]:
with open("../../data/solver.output.json") as f:
    data = json.load(f)

results = data["results"]
summary = data["summary"]

# Points where the solver hit the time limit may not be optimal
timeouts = [p for p in results if "maxTimeLimit" in p["termination_condition"]]

# A point is dominated (not efficient) if another point has a higher return
# for the same or lower CVaR
tol = 1e-9
dominated = [p for p in results
             if any(q["target_return"] > p["target_return"] and q["cvar"] <= p["cvar"] + tol
                    for q in results)]
efficient = [p for p in results if p not in dominated]

print(f"{len(results)} points, {len(efficient)} efficient, {len(dominated)} dominated, {len(timeouts)} timeouts")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))

ax.plot([p["cvar"] for p in efficient], [p["target_return"] for p in efficient],
        'o-', linewidth=2.5, markersize=5,
        label='Efficient frontier', color='#1f77b4', alpha=0.8)

if dominated:
    ax.plot([p["cvar"] for p in dominated], [p["target_return"] for p in dominated],
            'o', markersize=6, label='Dominated (not efficient)',
            color='#ff7f0e', alpha=0.9)

if timeouts:
    ax.plot([p["cvar"] for p in timeouts], [p["target_return"] for p in timeouts],
            'o', markersize=10, markerfacecolor='none', markeredgecolor='#d62728',
            markeredgewidth=2, label='Solver timeout (may be suboptimal)')

ax.set_xlabel('CVaR', fontsize=14, fontweight='bold')
ax.set_ylabel('Target Return (annual multiplier)', fontsize=14, fontweight='bold')
ax.set_title(f'Efficient Frontier (CVaR, alpha={summary["alpha"]}, K={summary["K"]}, J={summary["J"]})',
             fontsize=16, fontweight='bold', pad=20)

ax.grid(True, alpha=0.3, linestyle='--')
ax.legend(fontsize=12, loc='best', framealpha=0.95)
ax.set_facecolor('#f8f9fa')
fig.patch.set_facecolor('white')

plt.tight_layout()
plt.savefig('../../data/efficient_frontier.png', dpi=300, bbox_inches='tight')
plt.show()